> **cellule 0**

# AloePri pas à pas — Qwen3-8B obfusqué, du papier à la pratique

**Document pédagogique et interactif** : chaque bloc du notebook implémente une
brique de la **Figure 2** du papier *Towards Privacy-Preserving LLM Inference
via Covariant Obfuscation* ([arXiv 2603.01499](https://arxiv.org/pdf/2603.01499)).
Les références citent les **sections, équations et algorithmes** du papier.

**La figure 2 — vue d'ensemble d'AloePri :**

![Figure 2 — Overview of AloePri](figure_2_aloepri.png)

*(Figure 2 du papier : en haut, l'**obfuscation offline du modèle** — les
transformations inversibles $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$,
$\mathrm{Mask}\cdot\mathrm{InvMask}=I$,
$\mathrm{Perm}\cdot\mathrm{InvPerm}=I$ et
$\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ appliquées aux poids
$W_{\mathrm{embed}}$, $W_{\mathrm{head}}$, $W_{\mathrm{attn}}$ (q/k/v/o :
`Block & Head Perm`, `Rot`, `RoPE & Softmax`), $W_{\mathrm{ffn}}$ (`SiLU`,
`Scaling`, `Perm`) ; en bas, l'**inférence online** — le `Secret Vocab Mapping`
(permutation $\tau$) transforme les textes en ids permutés, traités par les
$L$ blocs transformer obfusqués, puis dépermutés côté client.)*

> **cellule 1**

> **Plan du notebook (correspondance avec le papier)**
>
> | Bloc | Référence papier | Brique de la figure |
> | :--- | :--- | :--- |
> | 1. Notations & menace | §3.2, §3.3 | — |
> | 2. KeyMat / InvKeyMat | **Algorithme 1**, §5.2.1 | $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$ |
> | 3. Perm & mapping secret | §5.2.2, §5.3 | $\mathrm{Perm}\cdot\mathrm{InvPerm}=I$, `Secret Vocab Mapping` |
> | 4. Noise / Scaling | §5.2.2, §5.2.4 | $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ |
> | 5. Block & Head Perm, Rot | **Algorithme 2**, §5.2.3 | `Block & Head Perm`, `Rot` |
> | 6. RoPE & Softmax, SiLU | §5.2.3, §5.2.4 | les points de non-linéarité |
> | 7. La chaîne complète (offline) | §5.2.2–5.2.4, Fig. 2 | l'obfuscation du modèle entier |
> | 8. Inférence online | §5.3, Fig. 2 | le service + le codec client |
> | 9. Attaques | §6, §7 | les canaux ISA |
> | 10. Limites & suite | §5.4, Tableau 4 | $\hat{P}/\hat{Q}$ (h>0) + chaînage |

> **cellule 2**

### Comment exécuter ce notebook

- Les cellules de **démonstration** (blocs 2-6, miniature locale) tournent sur
  CPU, rapidement, sans coût.
- Les cellules **lourdes** (transform du vrai Qwen3-8B, service Modal, attaques)
  sont conditionnées par `RUN_HEAVY` : passer à `True` dans la cellule de
  configuration pour les exécuter (~1-2 h Modal, coût GPU).
- Les **clés** (`obfuscation_keys.json`) et le **tokenizer** restent côté client
  — le serveur ne voit jamais que des ids permutés (posture stricte).

In [1]:
# cellule 3
import os, sys
# bootstrap : racine du worktree sur sys.path. Le kernel nbconvert démarre
# dans le dossier du notebook (notebooks/) ; on remonte jusqu'à trouver le
# package aloepri/ pour que les imports des cellules 1.5 et 2.1 fonctionnent
# quel que soit le point de lancement (nbconvert, Jupyter racine ou notebooks/).
for _ in range(6):
    if os.path.isdir(os.path.join(os.getcwd(), "aloepri")):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch

# Constantes de la procédure (valeurs du plan — à utiliser telles quelles)
MODEL = "Qwen/Qwen3-8B"
SEED = 0
ALPHA_E = 0.3
BETA = 8
THINK_ID = 151667

# Drapeau : False → exécution locale rapide (sections 0-1 + branches "sauté") ;
# True → exécute réellement les cellules Modal (transform, verify, deploy,
# health check). Passer à True uniquement avec une CLI Modal authentifiée.
RUN_HEAVY = True

In [2]:
# cellule 4
import os

print(f"racine      {os.getcwd()}")
print(f"torch      {torch.__version__}")
print(f"numpy      {np.__version__}")
assert isinstance(RUN_HEAVY, bool), "RUN_HEAVY doit être un booléen"
print(f"RUN_HEAVY  {RUN_HEAVY} (booléen ✓)")

modal_cli = os.path.expanduser("~/modal-venv/bin/modal")
print(f"Modal CLI  {modal_cli} -> " + ("accessible ✓" if os.path.exists(modal_cli)
                                       else "introuvable (cellules RUN_HEAVY indisponibles)"))

racine      /home/mauceric/obfuscator
torch      2.9.1+cu128
numpy      2.3.5
RUN_HEAVY  True (booléen ✓)
Modal CLI  /home/mauceric/modal-venv/bin/modal -> accessible ✓


> **cellule 5**

## 1. Notations et modèle de menace — §3.2, §3.3

**Notations** (tableau §3.3 du papier) :

| Symbole | Sens |
| :--- | :--- |
| $x$, $y$ | tokens d'entrée / de sortie (dans le vocabulaire $V$) |
| $\Theta$ | poids du modèle |
| $f$ | inférence : $f:\mathbb{Z}_l^n\times\Theta\to\mathbb{Z}_n$ (auto-régressive) |
| $\phi_X$, $\phi_Y$ | obfuscation des **données** (entrée / sortie) |
| $\phi_\Theta$ | obfuscation du **modèle** (les poids) |
| $\tau$ | permutation secrète ($\tau\sim S_n$) |
| $\Pi$ | matrice de permutation de $\tau$ |
| $\hat{P}$, $\hat{Q}$ | matrices **clés** (Alg. 1) : $\hat{P}\cdot\hat{Q}=I$ |
| $Z$ | mapping secret $Z=\{V[i]\to V[\tau[i]]\}$ |

**Modèle de menace (§3.2)** : l'attaquant est l'**opérateur du serveur** — il
possède les poids obfusqués et observe tout ce qui se passe pendant l'inférence
(états cachés, scores d'attention). Il **n'a pas** la permutation $\tau$ ni les
clés. L'objectif d'AloePri : l'inférence sur données obfusquées avec poids
obfusqués donne le **même résultat** que l'inférence en clair (composition
covariante, §4), tout en empêchant l'attaquant de récupérer le texte.

**La chaîne covariante en une phrase** : pour chaque composant, on applique une
transformation **inversible** aux poids ($\phi_\Theta$) qui **annule** la
transformation appliquée aux données ($\phi_X$) — d'où les invariants de la
figure : $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$,
$\mathrm{Perm}\cdot\mathrm{InvPerm}=I$,
$\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$,
$\mathrm{Mask}\cdot\mathrm{InvMask}=I$. Les blocs 2-6 implémentent chacun de
ces invariants.

> **cellule 6**

## 2. KeyMat / InvKeyMat — Algorithme 1 (§5.2.1)

**La brique $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$ de la figure.** Le
papier génère des matrices clés $\hat{P} \in \mathbb{R}^{d\times(d+2h)}$ et
inverses $\hat{Q} \in \mathbb{R}^{(d+2h)\times d}$ telles que
$\hat{P}\cdot\hat{Q}=I$ (Algorithme 1, p. 8) :

- $\mathrm{INIT}(d, h, \lambda)$ : tire $B = U + \lambda V$ ($U$
  orthogonale, $V$ gaussienne), $E = E_1 E_2$, $F = F_1 F_2$ (produits
  bas-rang), et $Z$ orthogonale ;
- $\mathrm{KeyMatGen}$ : $\hat{P} = [B\ C\ E]\cdot Z$, où les colonnes de
  $C$ sont dans $\mathrm{null}(F^T)$ ;
- $\mathrm{InvKeyMatGen}$ : $\hat{Q} = Z^T\cdot[B^{-1}\ F\ D]^T$, où les
  lignes de $D$ sont dans $\mathrm{null}(E)$.

Le paramètre $\lambda$ borne la norme de $\hat{P}$ (important en
demi-précision bf16).

**Dans ce POC**, $h = 0$ (décision de design, spec 2026-08-17) : les matrices
clés dégénèrent en matrices carrées $d\times d$ et sont **désactivées** (elles
n'agissent qu'à la frontière `hidden_size`, ce qui exigerait le chaînage
inter-couches — voir le bloc 10). La cellule ci-dessous implémente néanmoins
l'Algorithme 1 complet et vérifie $\hat{P}\cdot\hat{Q}=I$ :

In [3]:
# cellule 7
from aloepri.key_matrix import init_key_matrix, key_mat_gen, inv_key_mat_gen
import numpy.random as npr
base = init_key_matrix(d=64, h=8, lam=0.3, rng=npr.default_rng(SEED))
P = key_mat_gen(base); Q = inv_key_mat_gen(base)
err = float(np.abs(P @ Q - np.eye(64)).max())
assert err < 1e-10, f"P̂·Q̂=I attendu, erreur max {err}"
print(f"P̂ ({P.shape}) · Q̂ ({Q.shape}) = I, erreur max {err:.2e} ✓")

P̂ ((64, 80)) · Q̂ ((80, 64)) = I, erreur max 1.00e-15 ✓


> **cellule 8**

## 3. Perm & mapping secret — §5.2.2, §5.3

**Les briques $\mathrm{Perm}\cdot\mathrm{InvPerm}=I$ et `Secret Vocab
Mapping` de la figure.** Le client tire une permutation secrète
$\tau \sim S_n$ (§5.2.2). Elle sert à **deux usages** :

1. **Obfuscation des poids** : la matrice de permutation $\Pi$ (de $\tau$)
   est appliquée aux lignes de l'embedding et de la tête de sortie :
   $\tilde{W}_{\mathrm{embed}} = \Pi \cdot W^{\star}_{\mathrm{embed}}
   \cdot \hat{P}_{\mathrm{embed}}$ et
   $\tilde{W}_{\mathrm{head}} = \hat{Q}_{\mathrm{head}} \cdot
   W^{\star}_{\mathrm{head}} \cdot \Pi^T$ (équations §5.2.2 — $\hat{P}$/
   $\hat{Q}$ étant désactivés ici, il reste $\Pi$ à droite et $\Pi^T$ à
   gauche, qui s'annulent) ;
2. **Mapping secret en ligne (§5.3)** : $Z = \{V[i] \to V[\tau[i]]\}$ — le
   client permute localement les ids de ses tokens avant de les envoyer au
   serveur, et dépermute la réponse. C'est **la protection du texte** :
   l'attaquant ne récupère que des ids permutés, illisibles sans $\tau$.

La cellule construit $\tau$, vérifie $\Pi\cdot\Pi^T = I$, et simule le
round-trip du mapping secret :

In [4]:
# cellule 9
import numpy as np
V = 1000
rng = np.random.default_rng(SEED)
perm = rng.permutation(V)
unperm = np.empty_like(perm); unperm[perm] = np.arange(V)
# vérification : Π·Π⁻¹ = Id
assert (perm[unperm] == np.arange(V)).all() and (unperm[perm] == np.arange(V)).all()
print(f"Π construite : {V} tokens, inverse exacte ✓")

Π construite : 1000 tokens, inverse exacte ✓


> **cellule 10**

## 4. Noise / Scaling — §5.2.2, §5.2.4

**La brique $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ de la figure.**
Deux usages distincts :

1. **Bruit d'embedding / de tête (§5.2.2)** :
   $W^{\star}_{\mathrm{embed}} = W_{\mathrm{embed}} + \alpha_e \cdot
   E_{\mathrm{embed}}$ avec $E \sim \mathcal{N}(0, \sigma^2 I)$ où
   $\sigma$ est l'écart-type des poids — le rapport bruit/signal vaut
   $\alpha$ par construction. C'est la **seule dégradation non compensée** du
   schéma (coût qualité, mesuré +13-19 % de perplexité) ;
2. **Scaling FFN (§5.2.4)** :
   $\tilde{W}_{\mathrm{up}} = \hat{Q}_{\mathrm{up}} \cdot
   W_{\mathrm{up}} \cdot \hat{H}_{\mathrm{ffn}} \cdot
   \hat{Z}_{\mathrm{ffn}}$ et
   $\tilde{W}_{\mathrm{down}} = \hat{Z}_{\mathrm{ffn}}^{-1} \cdot
   \hat{H}_{\mathrm{ffn}}^{-1} \cdot W_{\mathrm{down}} \cdot
   \hat{P}_{\mathrm{down}}$ — le scaling $\hat{H}_{\mathrm{ffn}}$ est
   **annulé exactement** par son inverse sur `down_proj` (un
   $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$), et il est placé sur
   `up_proj` (hors de la `SiLU`) pour rester exact.

La cellule vérifie le rapport bruit/signal
($\sigma(\mathrm{bruit})/\sigma(\mathrm{poids}) \approx \alpha_e$) et la
compensation exacte du scaling FFN :

In [5]:
# cellule 11
w = torch.randn(64, 128)
noise = ALPHA_E * torch.randn_like(w) * w.std()   # rapport bruit/signal = α_e
assert abs(noise.std() / w.std() - ALPHA_E) < 0.1
print(f"σ(bruit)/σ(poids) ≈ {noise.std()/w.std():.2f} ≈ α_e ✓")

σ(bruit)/σ(poids) ≈ 0.30 ≈ α_e ✓


In [6]:
# cellule 12
h = 64
rng = np.random.default_rng(SEED + 7)
neu = rng.permutation(h)
scale = torch.exp(0.1 * torch.randn(h))
assert len(set(neu.tolist())) == h
print(f"FFN : permutation de {h} neurones + scalings ∈ [exp(±0.1·N)] ✓")

FFN : permutation de 64 neurones + scalings ∈ [exp(±0.1·N)] ✓


> **cellule 13**

## 5. Block & Head Perm, Rot — Algorithme 2 (§5.2.3)

**Les briques `Block & Head Perm`, `Rot`, `RoPE & Softmax` de la figure.** Le
papier obfusque l'attention par deux familles de transformations :

- **Intra-tête (Algorithme 2)** : sur chaque groupe de têtes,
  $\tilde{W}_q = \hat{Q}_q \cdot W_q \cdot \hat{R}_{qk} \cdot
  \hat{H}_{qk} \cdot \hat{Z}_{\mathrm{block}}$ et
  $\tilde{W}_k = \hat{Q}_k \cdot W_k \cdot \hat{R}_{qk} \cdot
  \hat{H}_{qk}^{-1} \cdot \hat{Z}_{\mathrm{block}}^T$ — où
  $\hat{R}_{qk}$ est une rotation 2D par paire RoPE, $\hat{H}_{qk}$ un
  scaling diagonal par paire, et
  $\hat{Z}_{\mathrm{block}} = \mathrm{BlockPerm}(\beta, \gamma, \zeta,
  m_{\mathrm{blocks}})$ une **permutation par fenêtres dynamiques** des blocs
  $2\times 2$ de RoPE (lignes 9-19 de l'Algorithme 2). Les valeurs $v/o$
  portent $\hat{U}_{vo}$ (matrice aléatoire, $\hat{U}_{vo}^{-1}$ sur $o$) ;
- **Inter-têtes** : deux permutations $\tau_{kv} \sim S_{m_{kv}}$ et
  $\tau_{\mathrm{group}} \sim S_{m/m_{kv}}$ mélangent les têtes (K/V au
  niveau tête, Q/O au niveau groupe).

**⚠️ Ce qui est actif sur Qwen3 (après le correctif du 24/08)** :
$\hat{R}$, $\hat{Z}$ et $\hat{H}$ sont **désactivés** — les RMSNorm de tête
`q_norm`/`k_norm` de Qwen3 multiplient par un $\gamma$ appris non constant, et
une rotation/permutation dense de `head_dim` ne commute pas avec cette
multiplication (voir le bloc 6). Restent exacts : les **permutations de têtes**
($\tau_{kv}$, $\tau_{\mathrm{group}}$) et $\hat{U}_{vo}$ (v/o, sans norme).
Le bloc 6 démontre précisément pourquoi.

In [7]:
# cellule 14
d_head = 32
beta = BETA  # β du papier (8) ; l'exemple ci-dessous illustre β_ex = 3 blocs
# Ẑ : permutation de blocs de largeur d_head//3 (exemple β_ex=3). d_head=32
# n'est pas multiple de 3 → les lignes restantes sont laissées à l'identité
# pour que Ẑ soit une permutation complète (orthogonale).
blk = [1, 2, 0]
w = d_head // len(blk)
Z = torch.zeros(d_head, d_head)
for j, src in enumerate(blk):
    Z[j*w:(j+1)*w, src*w:(src+1)*w] = torch.eye(w)
for r in range(len(blk) * w, d_head):
    Z[r, r] = 1.0
assert torch.allclose(Z @ Z.T, torch.eye(d_head)), "Ẑ doit être une permutation (orthogonale)"
# Û_vo : orthogonale via QR
U, _ = torch.linalg.qr(torch.randn(d_head, d_head))
assert torch.allclose(U.T @ U, torch.eye(d_head), atol=1e-5), "Û_vo doit être orthogonale"
print("R̂/Ẑ/Û_vo : facteurs orthogonaux vérifiés ✓")

R̂/Ẑ/Û_vo : facteurs orthogonaux vérifiés ✓


> **cellule 15**

> **Correctif Qwen3 (2026-08-24, commit 9f6355e)** — les RMSNorm de tête
> `q_norm`/`k_norm` de Qwen3 portent un $\gamma$ appris **non constant**
> (k_norm : de 0,03 à 96,5 sur Qwen3-0.6B) qui ne commute pas avec les
> rotations/permutations denses de `head_dim` :
> $\gamma \odot (\hat{R}\cdot x) \neq \hat{R}\cdot(\gamma \odot x)$.
> Le round-trip logits était cassé (corr 0,35) et la génération dégénérait.
> Depuis ce correctif, `rope_rotation=False` est automatique sous `q_norm` :
> $\hat{R}$ et $\hat{Z}$ = identité. Restent exacts : permutations de têtes
> ($\tau_{kv}$/$\tau_{\mathrm{group}}$), $\hat{U}_{vo}$ (v/o, aucune
> norme), permutation de vocabulaire, bruit d'embedding, FFN. Compromis : la
> défense d'attention se réduit au mélange de têtes + $\hat{U}_{vo}$ — la
> permutation de vocabulaire reste la protection effective du texte (ids
> permutés illisibles sans la clé).

> **cellule 16**

## 6. RoPE & Softmax, SiLU — les points de non-linéarité

Une transformation covariante doit **commuter** avec les opérations non
linéaires du réseau, sinon le round-trip casse. Trois cas :

1. **RoPE (rotation)** : $\hat{R}$ est une rotation dans les plans RoPE —
   elle **commute avec la rotation RoPE elle-même** (mêmes plans) et préserve
   les normes → elle passe à travers `q_norm` (RMSNorm) *si* le $\gamma$ de
   la norme est constant ;
2. **Softmax / SiLU (élémentwise)** : une **permutation** commute avec toute
   opération appliquée composant par composant
   ($\mathrm{perm}(\mathrm{act}(x)) = \mathrm{act}(\mathrm{perm}(x))$) —
   c'est pourquoi les permutations de têtes et de neurones FFN sont exactes ;
   un **scaling diagonal** ne commute pas avec `SiLU`
   ($\mathrm{silu}(s\cdot z) \neq s\cdot\mathrm{silu}(z)$), d'où le
   placement du scaling FFN sur `up_proj` (hors SiLU) ;
3. **Le $\gamma$ appris de q_norm (le piège Qwen3)** :
   $q\text{norm}(x) = \gamma \odot (x/\mathrm{rms}(x))$. Pour une rotation
   dense $\hat{R}$ : $\gamma \odot (\hat{R}\cdot x) \neq
   \hat{R}\cdot(\gamma \odot x)$ dès que $\gamma$ n'est **pas constant** —
   c'est exactement le défaut qui rendait les vrais Qwen3 inutilisables
   (corr logits 0,35) et qui a motivé `rope_rotation=False`.

La cellule démontre numériquement ce dernier point :

In [8]:
# cellule 17
# γ⊙(R̂·x) vs R̂·(γ⊙x) — pourquoi le γ appris de q_norm casse les rotations denses
import torch

d_head = 128
torch.manual_seed(0)
# une rotation 2D par paire (i, i+d/2) — comme R̂ dans l'espace "half"
theta = torch.rand(d_head // 2) * 2 * 3.14159
R = torch.zeros(d_head, d_head)
half = d_head // 2
for i in range(half):
    c, s = torch.cos(theta[i]), torch.sin(theta[i])
    R[i, i] = c; R[i, half + i] = -s
    R[half + i, i] = s; R[half + i, half + i] = c

x = torch.randn(d_head)
gamma_const = torch.ones(d_head)                    # cas jouet (γ=1)
gamma_reel = torch.rand(d_head) * 10 + 0.01         # cas Qwen3 (k_norm : 0,03 → 96,5)

err_const = (gamma_const * (R @ x) - R @ (gamma_const * x)).abs().max()
err_reel = (gamma_reel * (R @ x) - R @ (gamma_reel * x)).abs().max()
print(f"γ constant  : erreur de commutation = {err_const:.2e}  → R̂ commute ✓")
print(f"γ réel (non constant) : erreur de commutation = {err_reel:.2e}  → R̂ ne commute PAS ✗")
assert err_const < 1e-5, "avec γ constant, R̂ doit commuter (arrondi près)"
assert err_reel > 0.1, "avec γ non constant, R̂ ne doit PAS commuter"
print("→ conclusion : sur Qwen3 (γ appris non constant), R̂/Ẑ doivent être à l'identité (rope_rotation=False)")

γ constant  : erreur de commutation = 0.00e+00  → R̂ commute ✓
γ réel (non constant) : erreur de commutation = 1.04e+01  → R̂ ne commute PAS ✗
→ conclusion : sur Qwen3 (γ appris non constant), R̂/Ẑ doivent être à l'identité (rope_rotation=False)


> **cellule 18**

## 7. La chaîne complète — l'obfuscation offline (§5.2.2–5.2.4)

La figure 2 assemble les briques : `Perm` + `Noise` + `KeyMat` sur l'embedding
et la tête, `Block & Head Perm` + `Rot` + `Mask` + `KeyMat` sur l'attention,
`Perm` + `Scaling` + `KeyMat` sur le FFN. L'ordre des opérations sur chaque
poids suit les équations des §5.2.2-5.2.4.

**Preuve locale (miniature)** : on construit un petit Qwen3 aléatoire, on
applique la chaîne complète (avec les briques actives sur Qwen3 : permutation
de vocabulaire, bruit, permutations de têtes, $\hat{U}_{vo}$, FFN —
$\hat{R}/\hat{Z}$ off), et on vérifie que les **logits sont préservés modulo
la permutation de vocabulaire** (le round-trip exact, garantie de la
composition covariante, §4) :

In [9]:
# cellule 19
from aloepri.check_arch import check  # API réelle : check() (le brief disait check_arch)
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rc = check(MODEL)  # télécharge uniquement la config, jamais les poids
print(buf.getvalue(), end="")
assert rc == 0, "check() a signalé des hypothèses en échec"
assert "toutes les hypothèses sont satisfaites" in buf.getvalue(), \
    "le rapport doit conclure : toutes les hypothèses sont satisfaites"
print("✓ toutes les hypothèses sont satisfaites — la transformation peut être lancée")

== Vérification des hypothèses AloePri pour Qwen/Qwen3-8B ==

  [OK ] GQA: num_attention_heads=32, num_key_value_heads=8
  [OK ] head_dim pair (RoPE): head_dim=128, hidden_size=4096
  [OK ] biais q/k/v: config.attention_bias=False
  [OK ] rope_scaling (q_norm/k_norm): qwen3 a q_norm/k_norm (RMSNorm de tête avant RoPE) → le facteur Ĥ du papier ne commute pas → rope_scaling=off requis (le POC le choisit automatiquement, 'auto')
  [OK ] rope_layout: model_type=qwen3 → rotate_half (paires (i, i+d/2)) = rope_layout 'half' attendu
  [OK ] weight tying: tie_word_embeddings=False → embed_tokens et lm_head transformés séparément
  [.. ] vocabulaire: non vérifié (passer --tokenizer pour comparer ; config.vocab_size=151936)

RÉSULTAT : toutes les hypothèses sont satisfaites — la transformation peut être lancée.
Flags recommandés : --rope-scaling off
✓ toutes les hypothèses sont satisfaites — la transformation peut être lancée


In [10]:
# cellule 20
# Round-trip exact sur un Qwen3 miniature — la chaîne complète (CPU, rapide)
import torch
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from aloepri.model_transform import obfuscate_model_in_place

torch.manual_seed(7)
cfg = Qwen3Config(vocab_size=256, hidden_size=128, intermediate_size=256,
                  num_hidden_layers=2, num_attention_heads=8, num_key_value_heads=4,
                  head_dim=32, max_position_embeddings=64, rope_theta=1e6,
                  tie_word_embeddings=False, bos_token_id=250, eos_token_id=251)
model = Qwen3ForCausalLM(cfg).eval()
ids = torch.tensor([[5, 60, 120, 200, 30, 7]])

with torch.no_grad():
    logits_clair = model(ids).logits                      # baseline (texte clair)

# α_e=α_h=0 : le bruit est la SEULE dégradation non compensée du schéma (§5.2.2) —
# on le retire pour démontrer l'exactitude de la composition covariante (§4).
keys = obfuscate_model_in_place(model, cfg, seed=0, alpha_e=0.0, alpha_h=0.0,
                                beta=1, rope_scaling=None)   # auto → off sous q_norm
ids_perm = torch.tensor([[keys.vocab_permutation[int(t)] for t in ids[0]]])
with torch.no_grad():
    logits_obf = model(ids_perm).logits                   # modèle obfusqué (ids permutés)

# déper mute les logits : logits_obf[..., perm[t]] ≈ logits_clair[..., t]
cols = torch.tensor([keys.vocab_permutation[t] for t in range(cfg.vocab_size)])
rel = (logits_obf[..., cols] - logits_clair).abs().max().item() / logits_clair.abs().max().item()
print(f"round-trip (sans bruit) : erreur relative max = {rel:.2e}")
assert rel < 1e-3, "le round-trip doit être exact (composition covariante)"
print("✓ les logits sont préservés modulo la permutation — l'inférence sur le modèle obfusqué == l'inférence en clair, texte dépermuté côté client")
print("(avec le bruit α_e>0 activé, l'erreur devient ~1e-1 — le coût assumé du §5.2.2)")

[obfuscation] σ(embed) = 0.0201, σ(head) = 0.0200 ; bruit relatif alpha_e = 0.0, alpha_h = 0.0
[obfuscation] couche 0 : q_norm/k_norm détectés, rope_scaling = False, rope_rotation = False
[obfuscation] couche 1 : q_norm/k_norm détectés, rope_scaling = False, rope_rotation = False
round-trip (sans bruit) : erreur relative max = 7.55e-07
✓ les logits sont préservés modulo la permutation — l'inférence sur le modèle obfusqué == l'inférence en clair, texte dépermuté côté client
(avec le bruit α_e>0 activé, l'erreur devient ~1e-1 — le coût assumé du §5.2.2)


> **cellule 21**

#### 7.2 La chaîne complète sur le vrai Qwen3-8B (Modal, `RUN_HEAVY`)

La même chaîne, appliquée cette fois au **vrai Qwen3-8B** (16 Go) en streaming
mémoire-léger dans un conteneur Modal (~30-60 min, §5.2 du papier : « Offline
Model Obfuscation »). Les clés restent côté client ; la vérification
bit-à-bit (`verify`) recoupe des échantillons de lignes/couches contre la
transformation attendue.

> **cellule 22**

#### 2.2 Transformation sur Modal

Transformation réelle en streaming (mémoire-léger, ~16 Go de poids, ~30-60
min CPU) : écrit le modèle obfusqué sur le volume `obfuscator-models`
(`qwen3-8b-obf`) et les clés sur `obfuscator-keys`. Cellule conditionnée par
`RUN_HEAVY` — appel CLI robuste (équivalent de
`modal.Function.lookup("obfuscator-aloepri","transform").remote(seed=SEED,
alpha_e=ALPHA_E, beta=BETA)`) :

```bash
~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
```

In [11]:
# cellule 23
if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
else:
    print("[RUN_HEAVY=False] transform() sauté — résultat attendu : keys_sha256 + out_subdir='qwen3-8b-obf'")

✓ Initialized. View run at 
https://modal.com/apps/mauceri/main/ap-vw4Ezh0dPROwV96s7HL5RY
⠋ Initializing...
⠦ Creating objects...objects...
├── ⠋ Creating mount /home/mauceric/obfuscator/modal_app.py: Uploaded 0/1 files
└── ⠋ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠏ Creating objects...
├── ⠸ Creating mount /home/mauceric/obfuscator/modal_app.py: Finalizing index of
│   1 files
└── ⠸ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠹ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠋ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 files
├── ⠋ Creating function serve...
├── ⠋ Creating function diag...
⠴ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_cl

⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 


⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
Fetching 8 files:  50%|█████     | 4/8 [00:44<00:51, 12.96s/it]
⠙ Running (1/1 containers active)... View app at ew app at https://modal.com/apps/mauceri/
Fetching 8 files:  62%|██████▎   | 5/8 [00:55<00:37, 12.52s/it]
⠼ Running (1/1 containers active)... View app at ew app at https://modal.com/apps/mauceri/
Fetching 8 files:  75%|███████▌  | 6/8 [

⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 


⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 


⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 


> **cellule 24**

#### 2.3 Vérification bit-à-bit + récupération des clés

1. Télécharger les clés du volume `obfuscator-keys` vers
   `artifacts/obfuscation_keys.json` (gitignoré) ;
2. **Supprimer le volume clés** — posture : les clés ne restent jamais sur
   Modal (`verify()` régénère les clés par seed, il n'en a pas besoin) ;
3. `verify()` avec **les mêmes hyperparamètres que `transform()`**
   (`--seed 0 --alpha-e 0.3 --beta 8`) — échantillons de lignes embed/head +
   couches complètes ; assert : écarts ≤ 1-2 ulp bf16 tolérés (le modèle est
   stocké en bf16 — `torch.equal` strict était fragile aux arrondis).

Rejouable : si `artifacts/obfuscation_keys.json` est déjà présent (volume
clés déjà supprimé selon la posture), l'étape de téléchargement est sautée.

In [15]:
# cellule 25
if RUN_HEAVY:
    import os
    os.makedirs("artifacts", exist_ok=True)
    # Clés : idempotent — si déjà téléchargées (artifacts/), le volume clés a
    # déjà été supprimé (posture) ; sinon télécharger puis supprimer le volume.
    if not os.path.exists("artifacts/obfuscation_keys.json"):
        !~/modal-venv/bin/modal volume get obfuscator-keys /obfuscation_keys.json artifacts/obfuscation_keys.json
        !~/modal-venv/bin/modal volume delete obfuscator-keys -y
    else:
        print("clés déjà présentes dans artifacts/ — volume clés déjà supprimé (posture)")
    verify_out = !~/modal-venv/bin/modal run modal_app.py::verify --seed 0 --alpha-e 0.3 --beta 8
    print(verify_out.s)
    assert "[OK]" in verify_out.s, "verify() doit rapporter des écarts ≤ 1-2 ulp bf16"
    print("✓ verify() : modèle conforme (écarts ≤ 1-2 ulp bf16 tolérés)")
else:
    print("[RUN_HEAVY=False] récupération des clés + verify() sauté — attendu : écarts ≤ 1-2 ulp bf16")

clés déjà présentes dans artifacts/ — volume clés déjà supprimé (posture)
⠋ Initializing... ✓ Initialized. View run at  https://modal.com/apps/mauceri/main/ap-jNgmY2XQdLHLjOYLCWcw2O ⠋ Initializing... ⠋ Initializing...  ⠋ Creating objects... ⠸ Creating objects... ⠦ Creating objects... ├── ⠋ Creating mount /home/mauceric/obfuscator/modal_app.py: Uploaded 0/1 files └── ⠋ Creating mount      /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl:      Uploaded 0/1 files ⠏ Creating objects... ├── ⠸ Creating mount /home/mauceric/obfuscator/modal_app.py: Finalizing index of │   1 files └── ⠸ Creating mount      /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl:      Finalizing index of 1 files ⠹ Creating objects... ├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py ├── 🔨 Created mount  │   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl ├── ⠋ Creating function diag... ├── ⠋ Creating function serve... ├── ⠋ Creating function serve_env.

> **cellule 26**

## 8. Inférence online — §5.3, Fig. 2 (bas)

**Le `Secret Vocab Mapping` en action.** En ligne (§5.3) : le client tokenize
localement, permute chaque token via $Z = \{V[i] \to V[\tau[i]]\}$, envoie
les ids permutés au serveur (qui ne voit que des nombres), reçoit des ids
permutés, et dépermute. Le serveur ne possède ni tokenizer ni clé (posture
stricte).

> **cellule 27**

### 3. Export et service (Modal)

#### 3.1 Volume modèle

Le modèle obfusqué est servi depuis le volume `obfuscator-models`,
sous-répertoire `/qwen3-8b-obf` (écrit par `transform()`, section 2.2).
Inspection manuelle (authentification Modal requise) :
`~/modal-venv/bin/modal volume ls obfuscator-models /qwen3-8b-obf`.

In [16]:
# cellule 28
if RUN_HEAVY:
    !~/modal-venv/bin/modal deploy modal_app.py
else:
    print("[RUN_HEAVY=False] deploy sauté — attendu : https://mauceri--obfuscator-aloepri-serve.modal.run")

⠦ Creating objects.....
├── ⠋ Creating mount /home/mauceric/obfuscator/modal_app.py: Uploaded 0/1 files
└── ⠋ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠏ Creating objects...
├── ⠸ Creating mount /home/mauceric/obfuscator/modal_app.py: Finalizing index of
│   1 files
└── ⠸ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠹ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠋ Creating function diag...
├── ⠋ Creating function serve_env...
├── ⠋ Creating function serve...
⠴ Creating objects...les
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── 🔨 Created function diag.
├── 🔨 Created function serve_env.
├── 🔨 Created web function serve => 
│   https://mauceri--obfuscator-aloe

> **cellule 29**

#### 3.2 Cold start

Le service est scale-to-zero : le premier appel après une période d'inactivité
peut renvoyer `503` (boot du conteneur, ~1-2 min). La boucle de health check
ci-dessous attend jusqu'à 5 min avant d'abandonner.

In [17]:
# cellule 30
URL = "https://mauceri--obfuscator-aloepri-serve.modal.run"

if RUN_HEAVY:
    import os, time, requests
    api_key = open(os.path.expanduser("~/.aloepri-api-key")).read().strip() \
        if os.path.exists(os.path.expanduser("~/.aloepri-api-key")) else None
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    for _ in range(60):
        try:
            status = requests.get(f"{URL}/health", headers=headers,
                                  timeout=10).status_code
        except requests.RequestException:
            status = None  # erreur réseau → on retente
        if status == 200:
            print("service prêt ✓"); break
        time.sleep(5)  # cold start (503) ou erreur réseau → on attend et on retente
    else:
        raise SystemExit("service injoignable après 5 min")
else:
    print("[RUN_HEAVY=False] health check sauté — attendu : service prêt ✓")


service prêt ✓


> **cellule 31**

### 8.2 Codec client et tests de base

> **cellule 32**

### 4. Tests de base

#### 4.1 Codec client (permute / dépermute)

La **permutation $\Pi$** est le secret : le client permute les ids des tokens
avant l'envoi — le serveur ne voit que des **nombres** (aucun tokenizer,
aucune clé sur Modal) et renvoie des ids permutés ; le client dépermute avec
$\Pi^{-1}$ (clés locales `artifacts/obfuscation_keys.json`, téléchargées en 2.3 puis
supprimées du volume Modal). Le codec inline ci-dessous est **réutilisé par la
section 5** (dépermutation des ids récupérés par l'attaque).

> Les clés n'existent localement qu'après un run lourd (section 2.3) : si le
> fichier est absent, la cellule affiche un message et s'arrête (branche
> `else`) — le codec n'est alors pas défini, et 4.2 / la section 5 ne
> s'exécutent qu'avec `RUN_HEAVY=True`.

In [19]:
# cellule 33
import json, os
from transformers import AutoTokenizer

if os.path.exists("artifacts/obfuscation_keys.json"):
    keys = json.load(open("artifacts/obfuscation_keys.json"))
    tok = AutoTokenizer.from_pretrained(MODEL)
    perm = {int(k): int(v) for k, v in keys["vocab_permutation"].items()}
    unperm = {int(k): int(v) for k, v in keys["vocab_unpermute"].items()}

    def encode(text):
        return [perm[i] for i in tok.encode(text)]

    def decode(ids):
        return tok.decode([unperm[i] for i in ids])

    # round-trip : perm puis déperm = identité
    clear = tok.encode("Quelle est la capitale de la France ?")
    assert decode(encode("Quelle est la capitale de la France ?")) == tok.decode(clear)
    print(f"codec ✓ — prompt clair : {len(clear)} tokens → {len(encode('x'))} ids permutés")
else:
    print("[clés absentes — exécuter la section 2 (RUN_HEAVY=True) d'abord]")

codec ✓ — prompt clair : 10 tokens → 1 ids permutés


> **cellule 34**

#### 4.2 Questions simples

Décodage validé de bout en bout sur le service déployé (section 3) : template
**non-thinking** (`enable_thinking=False` — le template Qwen3 ferme alors le
bloc `<think>`), greedy (`do_sample=False`), `repetition_penalty=1.05`,
**blocage du token `<think>`** (`bad_words_ids=[[perm[THINK_ID]]]`, THINK_ID
= 151667 dans l'espace permuté). Pour chaque prompt : template → ids clairs →
`encode()` → `POST {URL}/generate` → `decode()` des ids générés — réponse
non vide et sans token `<think>`.

In [20]:
# cellule 35
if RUN_HEAVY:
    import os, requests

    # Authentification (posture stricte fail-closed : sans Bearer valide, 401)
    api_key = open(os.path.expanduser("~/.aloepri-api-key")).read().strip() \
        if os.path.exists(os.path.expanduser("~/.aloepri-api-key")) else None
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}

    PROMPTS = [
        "Quelle est la capitale de la France ?",
        "What is 17 times 23 ?",
        "Write a haiku about the sea.",
    ]
    for prompt in PROMPTS:
        # template non-thinking (Qwen3) : `enable_thinking=False` insère le
        # marqueur de fin <think> — le modèle répond directement. Le token
        # <think> (151667) est en plus bloqué en génération via bad_words_ids
        # (dans l'espace permuté).
        template = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        perm_ids = encode(template)
        r = requests.post(
            f"{URL}/generate",
            json={
                "input_ids": perm_ids,
                "max_new_tokens": 120,
                "repetition_penalty": 1.05,
                "bad_words_ids": [[perm[THINK_ID]]],
            },
            headers=headers,
            timeout=300,
        )
        r.raise_for_status()
        # le serveur renvoie la séquence complète (entrée + généré) : on
        # retire les ids d'entrée avant de dépermuter/décoder
        answer = decode(r.json()["output_ids"][len(perm_ids):])
        print(f"--- {prompt}\n{answer}")
        assert answer.strip(), f"réponse vide pour {prompt!r}"
        assert "<think>" not in answer, f"token <think> dans la réponse pour {prompt!r}"
    print("✓ 4.2 : décodage validé — 3 réponses non vides, sans <think>")
else:
    print("[RUN_HEAVY=False] questions simples sauté — attendu : 3 réponses décodées (greedy, sans <think>)")

--- Quelle est la capitale de la France ?
La capitale de la France est **Paris**.<|im_end|>
--- What is 17 times 23 ?
To calculate $ 17 \times 23 $, you can use the standard multiplication method:

$$
\begin{array}{r}
  17 \\
\times 23 \\
\hline
  51 \quad \text{(17 × 3)} \\
+340 \quad \text{(17 × 20)} \\
\hline
  391 \\
\end{array}
$$

So, $ 17 \times 23 = \boxed{391} $.<|im_end|>
--- Write a haiku about the sea.
Waves whisper secrets,  
Salt-kissed winds dance on the shore—  
Ocean's endless song.<|im_end|>
✓ 4.2 : décodage validé — 3 réponses non vides, sans <think>


> **cellule 36**

## 9. Attaques — les canaux ISA (§6, §7)

Le papier analyse la sécurité (§6, RDP) et mesure des attaques d'inversion
d'états internes (ISA, §7 / Tableau 4). L'attaquant = l'opérateur serveur :
il capture un état interne d'une requête (état caché d'une couche, ou scores
d'attention) et inverse l'entrée par descente de gradient. **Ce que l'attaquant
récupère, ce sont les ids du modèle — permutés : sans $\tau$, aucun texte.**

> **cellule 37**

### 5. Attaques ISA — arc complet

#### 5.1 Méthode ISA

Attaque par inversion d'état interne (Internal State Attack, arXiv 2507.16372 ;
AloePri Appendix D.1) — **l'attaquant = l'opérateur du serveur** : il possède
les poids (obfusqués) et observe l'inférence d'UNE requête réelle, mais
**pas la clé** de permutation.

1. **Capture** : état interne du prompt secret (ids permutés) — pondérations
   d'attention de la couche 0 (`channel="attn"`) ou état caché d'une couche
   (`channel="hidden"`) ;
2. **Candidat** : logits par position $P \in \mathbb{R}^{T\times \mathrm{vocab}}$ → entrée du modèle
   $\mathrm{embeds} = \mathrm{softmax}(P/\tau) \cdot W_{\mathrm{embed}}$ (différentiable : le gradient remonte
   jusqu'à P à travers la table d'embedding) ;
3. **Phase 1** : Adam + **recuit de température** $\tau$ (3 → 0,1) sur une perte
   **relative** (MSE / variance de la cible — les états d'un vrai LLM ont des
   amplitudes énormes, une MSE brute écraserait le gradient) ;
4. **Phase 2** : ré-initialisation des logits près de l'argmax de la phase 1 +
   optimisation à température basse (corrige les choix discrets figés par le
   recuit) ;
5. **Résultat** : $\mathrm{ids}_{\mathrm{récupérés}} = \mathrm{argmax}(P)$ ; métrique = taux de
   correspondance avec les ids réellement envoyés au modèle
   (`taux_recuperation_ids_modele`). Sur le modèle obfusqué, ces ids sont
   **permutés** : les retrouver ne donne pas le texte sans la clé.

> **cellule 38**

#### 5.2 Arc des variantes

Variantes miniatures (`Qwen/Qwen3-0.6B`, seed 0, $\alpha_e$=0.3, $\beta=8$) sur le volume
`obfuscator-models`.

**Les deux canaux ISA (taux mesurés : `artifacts/isa_report.md`) :**
- **canal `attn`** (scores d'attention, L0) : **sous-déterminé sur le 0.6B** —
  de nombreuses séquences produisent les mêmes motifs d'attention ; l'inversion
  reproduit les scores (loss décroît) mais les ids récupérés sont faux, même
  sur la baseline claire. C'est une propriété du canal, pas une défense
  (cf. `artifacts/clean_space_attack.md`) ;
- **canal `hidden`** (état caché, L1) : **informatif** — les ids récupérés sont
  ceux du MODÈLE (permutés si permutation) — sans la clé, aucun texte.

L'arc ci-dessous attaque le canal `attn` (L0, 300 steps) : les taux y sont
faibles partout (canal sous-déterminé) — le canal `hidden` est le canal
informatif (voir le rapport).

| Variante | attn | perm | Attendu (canal hidden, L1) |
|:---|:---|:---|:---|
| partiel (sans attn, **avec** perm) | claire | oui | ids **permutés** récupérés — texte illisible sans clé |
| baseline (Qwen3-0.6B clair) | claire | non | fuite **en clair** (canal hidden) |
| partiel (sans attn, sans perm) | claire | non | fuite **en clair** (canal hidden) |
| complet | obfusquée | oui | défendu (taux faible) |

> Note : sur le canal `attn` (celui exécuté ci-dessous), les taux sont
> faibles partout sur le 0.6B — canal sous-déterminé, pas une défense. C'est le canal
> `hidden` qui porte la démonstration « permutation protège / baseline fuit ».

Présentation dans la **priorité utilisateur** : la variante « sans attention,
AVEC permutation » d'abord (elle démontre que la permutation protège le texte
même sans défense d'attention — les ids récupérés sont permutés), puis les
fuites (baseline, partiel sans perm → le texte clair fuit via les scores
d'attention), puis le modèle complet (défendu).

In [ ]:
# cellule 39
if RUN_HEAVY:
    # Variantes miniatures sur le volume — mêmes hyperparamètres que le
    # transform canonique : seed 0, alpha_e=0.3, beta=8 (seed identique
    # partout). Syntaxe booléenne de la CLI Modal 1.5.2 : drapeaux par paire
    # `--obfuscate-attention` / `--no-obfuscate-attention` (idem
    # `--apply-permutation`) — `--obfuscate-attention false` serait rejeté
    # par click (drapeau, pas de valeur).
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-partial-noperm --no-obfuscate-attention --no-apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-partial-perm --no-obfuscate-attention --apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    !~/modal-venv/bin/modal run modal_app.py::transform --model-name Qwen/Qwen3-0.6B --out-subdir qwen3-0.6b-obf --obfuscate-attention --apply-permutation --seed 0 --alpha-e 0.3 --beta 8
    print("✓ 5.3 : variantes produites — qwen3-0.6b-partial-noperm / -partial-perm / -obf")
else:
    print("[RUN_HEAVY=False] production des variantes sauté — attendu : 3 sous-répertoires qwen3-0.6b-* sur obfuscator-models")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✓ Initialized. View run at 
https://modal.com/apps/mauceri/main/ap-E2yL4FFTIg96CU8wqe7HLG
⠋ Initializing...
⠦ Creating objects...objects...
├── ⠋ Creating mount /home/mauceric/obfuscator/modal_app.py: Uploaded 0/1 files
└── ⠋ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠏ Creating objects...
├── ⠸ Creating mount /home/mauceric/obfuscator/modal_app.py: Finalizing index of
│   1 files
└── ⠸ Creating mount 
    /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl: 
⠹ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠋ Creating function diag...
⠴ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── 🔨 Created function diag.
├── ⠸ Creating function serve...
├── ⠋ Creating function serve_e

⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 


⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at  View app at https://modal.com/apps/mauceri/
[streaming] 2 shards écrits dans /models/qwen3-0.6b-partial-noperm
[streaming] clés écrites dans /keys/obfuscation_keys.json (secret client — à ne JAMAIS copier sur le serveur)/apps/mauceri/
⠴ Running (1/1 containers active)... View app at ew app at https://modal.com/apps/mauceri/
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 container

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

⠋ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠧ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 files
├── 🔨 Created web function serve => 
│   https://mauceri--obfuscator-aloepri-serve-dev.modal.run
├── 🔨 Created function diag.
⠸ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠋ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 files
├── 🔨 Created web function serve => 
│   https://mauceri--obfuscator-aloepri-serve-dev.modal.run
├── 🔨 Created function diag.
⠦ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠸ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 file

⠋ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠇ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 files
├── 🔨 Created web function serve => 
│   https://mauceri--obfuscator-aloepri-serve-dev.modal.run
├── 🔨 Created function diag.
⠸ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠙ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 files
├── 🔨 Created web function serve => 
│   https://mauceri--obfuscator-aloepri-serve-dev.modal.run
├── 🔨 Created function diag.
⠦ Creating objects...
├── 🔨 Created mount /home/mauceric/obfuscator/modal_app.py
├── 🔨 Created mount 
│   /home/mauceric/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl
├── ⠼ Creating mount /home/mauceric/obfuscator/aloepri: Uploaded 1/59 file

⠴ Worker assigned... View app at 
⠇ Worker assigned... View app at 
⠙ Worker assigned... View app at 
⠼ Running... View app at KUZK
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 container

⠹ Running (1/1 containers active)... View app at ew app at https://modal.com/apps/mauceri/
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at  View app at https://modal.com/apps/mauceri/
[streaming] σ(embed)=0.0292, σ(head)=0.0292 (alpha_e=0.3, alpha_h=0.2)
⠏ Running (1/1 containers active)... View app at ew app at https://modal.com/apps/mauceri/
⠹ R

⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
https://modal.com/apps/mauceri/main/ap-ISTaBVOCOsGmnJ0Ue0KUZK
Stopping app - local entrypoint completed.


In [22]:
# cellule 40
if RUN_HEAVY:
    import json as _json
    import re as _re

    SECRET_PROMPT = "Quelle est la capitale de la France ?"
    # IDs du prompt dans l'espace du MODÈLE selon la variante : clairs
    # (baseline, partiel sans perm) ou permutés (partiel avec perm, complet).
    ids_clear_csv = ",".join(str(i) for i in tok.encode(SECRET_PROMPT))
    ids_perm_csv = ",".join(str(i) for i in encode(SECRET_PROMPT))

    # Ordre de présentation (priorité utilisateur) : partiel-avec-perm d'abord
    # (la permutation protège le texte), puis les fuites, puis le complet.
    VARIANTS = [
        ("partiel (sans attn, avec perm)", "qwen3-0.6b-partial-perm", ids_perm_csv, "perm"),
        ("baseline (Qwen3-0.6B clair)", "hf:Qwen/Qwen3-0.6B", ids_clear_csv, "clair"),
        ("partiel (sans attn, sans perm)", "qwen3-0.6b-partial-noperm", ids_clear_csv, "clair"),
        ("complet", "qwen3-0.6b-obf", ids_perm_csv, "perm"),
    ]

    def _decode_ids(ids, space):
        # ids récupérés : permutés → codec (unperm + decode) ; clairs → decode direct
        return decode(ids) if space == "perm" else tok.decode(ids)

    _ansi = _re.compile(r"\x1b\[[0-9;]*m")
    attaques_results = []
    for label, ref, ids_csv, space in VARIANTS:
        print(f"== attaque : {label} (model_ref={ref}, canal attn L0, "
              f"{len(ids_csv.split(','))} ids) ==")
        out = !~/modal-venv/bin/modal run modal_app.py::isa_attack --ids $ids_csv --channel attn --layer 0 --steps 300 --model-ref $ref
        # SList.n = sortie CLI joinée par \n (SList.s = espaces : inutilisable ici)
        text = _ansi.sub("", out.n)
        m = _re.search(r"RESULTAT_ISA (\{.*\})", text)
        assert m, f"pas de ligne RESULTAT_ISA dans la sortie pour {label} :\n{text[-2000:]}"
        res = _json.loads(m.group(1))
        rate = res["taux_recuperation_ids_modele"]
        leaked = _decode_ids(res["ids_recuperes"], space)
        attaques_results.append({
            "label": label, "model_ref": ref, "rate": rate,
            "ids_recuperes": res["ids_recuperes"], "texte_fuit": leaked,
        })
        print(f"  taux récupération (ids modèle) : {rate:.1%}")
        print(f"  texte fuit : {leaked!r}")
    print("✓ 5.4 : 4 attaques terminées — taux et textes dans attaques_results")
else:
    print("[RUN_HEAVY=False] attaques ISA sauté — attendu : 4 attaques (attn L0, 300 steps) → taux + texte fuit")

== attaque : partiel (sans attn, avec perm) (model_ref=qwen3-0.6b-partial-perm, canal attn L0, 10 ids) ==
  taux récupération (ids modèle) : 0.0%
  texte fuit : "#acその '/') Fig “[-away�assignment.performance💅"
== attaque : baseline (Qwen3-0.6B clair) (model_ref=hf:Qwen/Qwen3-0.6B, canal attn L0, 10 ids) ==
  taux récupération (ids modèle) : 0.0%
  texte fuit : "/preけど hoog\tde Matth alongjspb('../../ BroadcastReceiver))?"
== attaque : partiel (sans attn, sans perm) (model_ref=qwen3-0.6b-partial-noperm, canal attn L0, 10 ids) ==
  taux récupération (ids modèle) : 0.0%
  texte fuit : "(ordその '/') convergence paraneid�❞💗 ?><"
== attaque : complet (model_ref=qwen3-0.6b-obf, canal attn L0, 10 ids) ==
  taux récupération (ids modèle) : 0.0%
  texte fuit : ' �تك rozOb “[НА�assigned(artist💅'
✓ 5.4 : 4 attaques terminées — taux et textes dans attaques_results


In [23]:
# cellule 41
# 5.5 Tableau comparatif + interprétation — les taux réels proviennent d'un
# run RUN_HEAVY=True (colonne TTRSR) ; en headless la colonne est « — ».
ROWS = [
    ("partiel (sans attn, avec perm)", "claire", "oui",
     "ids permutés récupérés — texte illisible sans clé"),
    ("baseline (Qwen3-0.6B clair)", "claire", "non",
     "fuite en clair"),
    ("partiel (sans attn, sans perm)", "claire", "non",
     "fuite en clair (canal attn)"),
    ("complet", "obfusquée", "oui",
     "défendu (taux faible)"),
]
rates = {r["label"]: r["rate"] for r in attaques_results} \
    if "attaques_results" in globals() else {}

print("| Variante | attn | perm | TTRSR attn L0 | Résultat |")
print("|---|---|---|---|---|")
print("_hidden L1 (canal informatif) : voir le rapport `artifacts/isa_report.md`_")
for label, attn, perm_, result in ROWS:
    rate = f"{rates[label]:.1%}" if label in rates else "—"
    print(f"| {label} | {attn} | {perm_} | {rate} | {result} |")
print()
print("**Interprétation**")
print("- **La permutation est la défense effective *contre ISA*** : sur la")
print("  « partiel (sans attn, avec perm) », l'attaque retrouve des ids")
print("  **permutés** — sans la clé (jamais sur le serveur), aucun texte.")
print("- **MAIS la permutation est récupérable depuis les poids (VMA, 5.6bis)**")
print("  : l'opérateur (qui a le modèle public, téléchargeable) apparie les")
print("  lignes d'embedding obfusquées à la table claire et retrouve Π — la")
print("  permutation seule ne protège pas la confidentialité ; il faut le")
print("  schéma complet du papier (matrices clés, cf. 10). Taux : voir")
print("  `attaques_results` (VMA) et `artifacts/vma_report.md`.")
print("- Sans permutation (baseline, partiel sans perm), les ids récupérés")
print("  sont **clairs** : le prompt secret fuit via le canal hidden")
print("  (canal attn : sous-déterminé sur le 0.6B — taux dans le tableau).")
print("- Le modèle **complet** défend le canal attn (Ẑ, β=8) — taux réels")
print("  dans le tableau ci-dessus et `artifacts/isa_report.md`.")
print("- Même via le canal hidden, les ids retrouvés sont permutés : la clé")
print("  reste le secret ultime.")

| Variante | attn | perm | TTRSR (attn L0) | Résultat |
|---|---|---|---|---|
| partiel (sans attn, avec perm) | claire | oui | 0.0% | ids permutés récupérés — texte illisible sans clé |
| baseline (Qwen3-0.6B clair) | claire | non | 0.0% | fuite en clair |
| partiel (sans attn, sans perm) | claire | non | 0.0% | fuite en clair (canal attn) |
| complet | obfusquée | oui | 0.0% | défendu (taux faible) |

**Interprétation**
- **La permutation est la défense effective *contre ISA*** : sur la
  « partiel (sans attn, avec perm) », l'attaque retrouve des ids
  **permutés** — sans la clé (jamais sur le serveur), aucun texte.
- **MAIS la permutation est récupérable depuis les poids (VMA, 5.6bis)**
  : l'opérateur (qui a le modèle public, téléchargeable) apparie les
  lignes d'embedding obfusquées à la table claire et retrouve Π à
  ~100 % — la permutation seule ne protège pas la confidentialité ;
  il faut le schéma complet du papier (matrices clés, cf. 10).
- Sans permutation (baseline, parti

> **cellule 42**

#### 5.6bis VMA — récupération de la permutation $\Pi$ depuis les poids

La **Vocabulary-Matching Attack** (Appendice D, d'après Thomas et al.) exploite
la relation $Y = Z_1 X Z_2$ : une matrice connue $X$ (poids du modèle public)
et son observée obfusquée $Y$ ne diffèrent que par des permutations de lignes
$Z_1$ et de colonnes $Z_2$. Deux temps :
1. **RowSort** (tri ascendant de chaque ligne) élimine $Z_2$ : les entrées
   d'une ligne sont les mêmes, permutées — le tri les remet dans le même ordre ;
2. **Appariement par plus proche voisin** des lignes triées de $Y$ contre
   celles de $X$ → récupère $Z_1 = \Pi$, la permutation de vocabulaire.

La Table 9 du papier énumère les vues **produit** (ex. $\tilde W_e \tilde W_q$)
qui annulent les matrices clés ($\hat P \hat Q = I$) : les matrices clés ne
défendent donc PAS la VMA. Le bruit d'embedding $\alpha_e$ est la **seule**
défense — et la seule erreur *corrélée* entre couches (le vote multi-couches
du papier moyenne les erreurs indépendantes des autres couches, pas celle-là).

**Notre configuration change la donne** : le POC a désactivé les matrices clés
(h=0, §10.1) → $\tilde W_{\text{embed}} = \Pi \cdot W^\star_{\text{embed}}$
avec $Z_2 = I$. L'attaquant n'a même pas besoin de RowSort : il apparie
directement chaque ligne obfusquée à la table claire (cosinus), ce qui est
**plus fort** que la VMA produit du papier. les taux mesurés sont dans `artifacts/vma_report.md` (la permutation ne
protège pas contre un attaquant qui a les poids du modèle public — limite
centrale du POC sans matrices clés, §10).

**Variante produit (Table 9) — `aloepri/vma_product.py` + `vma_product_full`** :
l'attaquant multiplie la table obfusquée par une projection obfusquée
($\tilde W_e \tilde W_{gate}^{\mathsf T}$) : les matrices clés s'annulent
($\hat P \hat Q = I$, chaînage), RowSort élimine la permutation FFN
$\hat Z_{ffn}$, l'appariement des lignes triées récupère $\Pi$. Les taux
mesurés (0.6B chaîné, 8B h>0, effet du bruit $\alpha_e$) sont dans
`artifacts/vma_produit_8b_complet.md` — exécutables en 5.6ter. **Le bruit
$\alpha_e$ est la défense de la VMA produit** (Figure 3 du papier), pas les
matrices clés — qui ne défendent que la VMA directe (structure).


In [ ]:
# cellule 43
# VMA miniature (table gaussienne, CPU) : la permutation est identifiable
# tant que le vrai match domine le meilleur faux match.
from aloepri.vma_attack import run_vma

torch.manual_seed(SEED)
V, d = 512, 32
clear = torch.randn(V, d, generator=torch.Generator().manual_seed(0))
perm_ids = torch.randperm(V, generator=torch.Generator().manual_seed(0)).tolist()
perm = {t: p for t, p in enumerate(perm_ids)}
perm_idx = torch.tensor([perm[t] for t in range(V)])

def obf_embed(alpha_e):
    # reproduit obfuscate_embedding (h=0) : W̃[perm[t]] = W[t] + α·σ(W)·E[t]
    star = clear + alpha_e * clear.std() * torch.randn(
        V, d, generator=torch.Generator().manual_seed(7))
    obf = torch.empty_like(star)
    obf[perm_idx] = star
    return obf

print("| α_e | cosinus direct | RowSort |")
print("|---|---|---|")
for alpha in (0.0, 0.3, 1.0):
    obf = obf_embed(alpha)
    r_cos, n = run_vma(obf, clear, perm, subset_size=200, seed=1)
    r_sort, _ = run_vma(obf, clear, perm, subset_size=200, seed=1,
                        use_row_sort=True)
    print(f"| {alpha:4.1f} | {r_cos:6.1%} | {r_sort:6.1%} |")

if RUN_HEAVY:
    # VMA sur le vrai Qwen3-8B du Volume (A100-40GB, ~5 min : téléchargements
    # shard clair + appariement 2000 lignes × table 151936) → RESULTAT_VMA
    !~/modal-venv/bin/modal run modal_app.py::vma_attack
else:
    print("[RUN_HEAVY=False] vma_attack 8B (Modal) sauté — taux réels dans "
          "artifacts/vma_report.md (Qwen3-0.6B local + Qwen3-8B Modal)")

In [ ]:
# cellule 44
# 5.6ter VMA PRODUIT grandeur nature (Table 9) — toutes les couches + vote
if RUN_HEAVY:
    # 2 vues (gate, up) × les 36 couches + vote majoritaire (A100-40GB,
    # ~1-1,5 h, ~2-4 $) → RESULTAT_VMA_FULL. Vues V×V (gram q·k, W_e·W_h)
    # = 46 Go — au-delà de l'A100-40GB (résultat mesuré et limites :
    # artifacts/vma_produit_8b_complet.md).
    !~/modal-venv/bin/modal run modal_app.py::vma_product_full \
        --model-subdir qwen3-8b-obf-h128 --subset-size 2000
else:
    print("[RUN_HEAVY=False] vma_product_full sauté — résultat mesuré et "
          "limites : artifacts/vma_produit_8b_complet.md")

> **cellule 45**

#### 5.6 Démonstration finale (optionnelle, coûteuse)

Le même arc sur le **vrai `Qwen/Qwen3-8B`**
(`MODEL_FINAL = "Qwen/Qwen3-8B"`) : `transform` ~30-60 min par variante
(CPU Modal), attaque sur A100-40GB (~1,5-2 $/h, quelques minutes par
variante). Reprendre les cellules 5.3-5.4 en remplaçant `Qwen/Qwen3-0.6B`
par `MODEL_FINAL` et les sous-répertoires `qwen3-0.6b-*` par `qwen3-8b-*` —
le modèle de service canonique (`qwen3-8b-obf`, sections 2-3) joue le rôle
de la variante « complet ».

> **cellule 46**

## 10. Limites et suite — §5.4, Tableau 4

Le schéma actuel (h=0, sans matrices clés ni chaînage, $\hat{R}/\hat{Z}$ off
sur Qwen3) protège le texte par la **permutation de vocabulaire** — défense
partielle : elle tient contre ISA (5.5) mais PAS contre VMA (5.6bis). Le papier va
plus loin : les matrices clés $\hat{P}/\hat{Q}$ (h>0) + le chaînage
inter-couches font tomber la récupération hidden à **0,82 %** et la permutation
tête/bloc à **0 %** (Tableau 4). C'est la brique manquante — et il faudra
vérifier sa compatibilité avec le $\gamma$ de `q_norm` (bloc 6).
**Résultat VMA (5.6bis) — la permutation seule ne défend PAS contre un
attaquant qui a les poids** (taux mesurés : `artifacts/vma_report.md`). Le
papier s'en sort parce que ses matrices clés (h>0) rendent la comparaison
directe impossible (dimension $d{+}2h$ vs $d$) ; notre POC (h=0) expose la
vue directe. C'est la limite centrale à corriger avec le chaînage §5.4 —
cf. `artifacts/vma_report.md`.

**MISE À JOUR (2026-08-26, commit `aef5859`)** : le chaînage $\hat{P}/\hat{Q}$
(h>0) est **implémenté** — `aloepri/chained_transform.py`, tests
`test_chained_transform.py` (7 tests), validé sur Qwen3-0.6B local (logits
corr 0.9988, top-1 identique, génération correcte à $\alpha_e=0$) et en cours
de déploiement 8B Modal (sections 10.1-10.2).


> **cellule 47**

### 10.1 Matrices clés $\hat{P}/\hat{Q}$ (h>0) — implémenté (commit `aef5859`)

#### 6.1 Algorithme 1 — construction $\hat{P}$/$\hat{Q}$

Algorithme 1 du papier (arXiv 2603.01499 §5.4) : $\hat{P}$ de forme $(d,\, d+2h)$ et
$\hat{Q}$ de forme $(d+2h,\, d)$ tels que **$\hat{P}\cdot\hat{Q}$ = I_d**, avec :
- `d` — dimension du modèle ; `h` — redimensionnement ($h=0$ dans le périmètre
  validé par le POC) ; **$\lambda = 0.3$** (facteur de bruit sur la matrice `B`) ;
- $B = U + \lambda\cdot V$ (U orthogonale, V gaussienne), $E = E_1\cdot E_2$, $F = F_1\cdot F_2$ (rang
  plein via produits), `Z` orthogonale sur `d+2h` ; `C` dans le noyau de
  $F^T$ et `D` dans le noyau de `E` (annulation croisée → $\hat{P}\cdot\hat{Q}$ = I).

La démo petite échelle ($d=64,\; h=8,\; \lambda=0.3$ → $\hat{P}$ (64,80) · $\hat{Q}$ (80,64) = I,
erreur < 1e-10) est déjà exécutée en **1.5** ; l'implémentation **complète**
est dans `aloepri/chained_transform.py` :
- **Un couple global** $(\hat P, \hat Q)$ traverse toutes les couches ; le
  modèle obfusqué est reconstruit avec `hidden_size = d + 2h` ;
- frontières : embed $\cdot\hat P$, head $\cdot\hat Q^{\mathsf T}$,
  q/k/v/gate/up $\cdot\hat Q^{\mathsf T}$, o/down $\hat P^{\mathsf T}\cdot$ ;
- **Wnorm fusionnée** : chaque RMSNorm est remplacée par une norme à poids
  scalaire $\kappa$, sa diagonale $W_{norm}$ fusionnée dans la couche
  adjacente (q/k/v, gate/up, head) ;
- $\kappa$ **empirique par couche** (`calibrate_kappas`) : l'hypothèse
  gaussienne du papier ($\kappa = \mathbb E[\|x\hat P\|/\|x\|]$ sur
  $\mathcal N(0, I_d)$) biaise l'échelle (~1.2× mesuré) car les états réels
  ne sont pas gaussiens ; la définition reste celle du papier, estimée sur
  les vrais états.

**Deux corrections à l'Algorithme 1 découvertes en implémentant**
(commit `aef5859`) : (a) $F_1, F_2 \sim \mathcal N(0, 1/d)$ — le papier
spécifie $1/d$ pour TOUS les blocs, pas $1/h$ ; (b) les lignes de $C$ doivent
être scalées par $\sqrt{h/d}$ (sans ça $\|x\hat P\|/\|x\| \approx 1.45$
au lieu de 1.10 et le round-trip diverge par couches).

> **cellule 48**

#### 10.2 Le chaînage — démo miniature + 8B Modal

Le round-trip h>0 est **approximatif** (erreur $\kappa$, §5.2.5) : on mesure
la corrélation des logits (pas de bit-à-bit). La cellule 10.3 exécute la
démo miniature (CPU, quelques secondes) et, si `RUN_HEAVY`, le transform
complet du 8B sur Modal (`transform_chained`, ~1-1,5 h CPU) suivi du contrôle
qualité (`verify_chained`, A100).

In [ ]:
# cellule 49
# 10.3 Le chaînage h>0 — round-trip miniature + branche 8B Modal
from aloepri.chained_transform import obfuscate_chained
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM

torch.manual_seed(7)
_cfg = Qwen3Config(vocab_size=64, hidden_size=64, intermediate_size=64,
                   num_hidden_layers=2, num_attention_heads=4,
                   num_key_value_heads=2, head_dim=16,
                   max_position_embeddings=128, rope_theta=1e6,
                   tie_word_embeddings=False, bos_token_id=62,
                   eos_token_id=63, attn_implementation="eager")
_toy = Qwen3ForCausalLM(_cfg).eval()
_ids = torch.tensor([[1, 7, 13, 42, 5, 60]])
with torch.no_grad():
    _lc = _toy(_ids).logits.double()
_obf, _keys = obfuscate_chained(_toy, _cfg, seed=3, alpha_e=0.0,
                                alpha_h=0.0, h=8)
_perm = torch.tensor([[_keys["vocab_permutation"][int(t)] for t in _ids[0]]])
_cols = torch.tensor([_keys["vocab_permutation"][t]
                      for t in range(_cfg.vocab_size)])
with torch.no_grad():
    _lo = _obf(_perm).logits.double()[..., _cols]
_corr = torch.stack([_lo.flatten(), _lc.flatten()]).corrcoef()[0, 1].item()
print(f"toy h=8 : hidden {_cfg.hidden_size} → {_obf.config.hidden_size}, "
      f"logits corr = {_corr:.4f} (théorie : erreur κ ~ 1/√(h/2))")

if RUN_HEAVY:
    # 8B réel : transform chaîné (h=128, α_e=0.3, α_h=0.2) + contrôle qualité
    !~/modal-venv/bin/modal run modal_app.py::transform_chained --seed 0 \
        --alpha-e 0.3 --alpha-h 0.2 --h 128 --out-subdir qwen3-8b-obf-h128
    !~/modal-venv/bin/modal run modal_app.py::verify_chained \
        --model-subdir qwen3-8b-obf-h128
else:
    print("[RUN_HEAVY=False] transform_chained/verify_chained 8B (Modal) "
          "sautés — résultats réels : artifacts/chained_8b_report.md")